In [5]:
# ============================================================
# SMART COTTON AI — COMPLETE SYSTEM
# Autoencoder + CNN + Arduino + Telegram + Image Alerts
# Webcam + Detection History + Colorful UI
# ============================================================

# !pip install ultralytics pyserial gradio opencv-python tensorflow pillow pandas

import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import gradio as gr
import requests
import serial
import time
import pandas as pd
from datetime import datetime
import tempfile

# ---------------- SETTINGS ----------------
settings = {
    "telegram_bot_id":"8552268721:AAFP959y25shDht3xrRqmwmmmTygIHZmQVo",  # Replace with your bot token
    "telegram_chat_id":"",
    "arduino_com_port":"COM3"
}

history=[]
arduino=None

# ---------------- LOAD MODELS ----------------
print("Loading AI models...")
autoencoder = load_model(
    r"C:\Users\muthu\Desktop\Cotton_Bollworm_Project\autoencoder_model\bollworm_autoencoder.h5",
    compile=False
)
cnn = load_model("cotton_disease_mobilenet.h5", compile=False)
print("Models Loaded ✔")

# ---------------- CLASS NAMES ----------------
class_names=['aphids','army_worm','bacterial_blight','fungal_disease','healthy','powdery_mildew']

# ---------------- FERTILIZER ----------------
fertilizer={
    "aphids":{"english":"Spray Imidacloprid insecticide","tamil":"இமிடாக்ளோபிரிட் தெளிக்கவும்"},
    "army_worm":{"english":"Apply Chlorantraniliprole","tamil":"குளோரான்ட்ரானிலிப்ரோல் பயன்படுத்தவும்"},
    "bacterial_blight":{"english":"Spray Copper oxychloride","tamil":"காப்பர் ஆக்ஸிகுளோரைடு தெளிக்கவும்"},
    "fungal_disease":{"english":"Apply Carbendazim","tamil":"கார்பெண்டசிம் பயன்படுத்தவும்"},
    "powdery_mildew":{"english":"Spray Sulfur fungicide","tamil":"சல்பர் தெளிக்கவும்"},
    "healthy":{"english":"Plant healthy","tamil":"பயிர் ஆரோக்கியமாக உள்ளது"}
}

# ---------------- ARDUINO ----------------
def connect_arduino():
    global arduino
    try:
        arduino = serial.Serial(settings["arduino_com_port"], 9600, timeout=1)
        time.sleep(3)  # Allow Arduino to reset
        arduino.flush()
        print("Arduino Connected ✔")
    except Exception as e:
        print("Arduino not connected:", e)

def trigger_spray(level="TRIGGER_LOW"):
    if arduino is None:
        print("Arduino not connected")
        return
    try:
        if level=="TRIGGER_HIGH":
            arduino.write(b"TRIGGER_HIGH\n")
        elif level=="TRIGGER_LOW":
            arduino.write(b"TRIGGER_LOW\n")
        elif level=="RESET":
            arduino.write(b"RESET\n")
        print("SPRAY COMMAND:", level)
    except Exception as e:
        print("Arduino command failed:", e)

# ---------------- TELEGRAM ----------------
def send_telegram(message, image=None):
    if settings["telegram_chat_id"]=="":
        return
    try:
        if image is not None:
            temp_file = tempfile.NamedTemporaryFile(suffix=".jpg", delete=False)
            cv2.imwrite(temp_file.name, cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
            url = f"https://api.telegram.org/bot{settings['telegram_bot_id']}/sendPhoto"
            with open(temp_file.name,'rb') as img:
                requests.post(url, data={
                    "chat_id":settings["telegram_chat_id"],
                    "caption":message
                }, files={"photo":img})
        else:
            url = f"https://api.telegram.org/bot{settings['telegram_bot_id']}/sendMessage"
            requests.post(url, data={
                "chat_id":settings["telegram_chat_id"],
                "text":message
            })
        print("Telegram Sent ✔")
    except Exception as e:
        print("Telegram error:", e)

# ---------------- PREPROCESS ----------------
def preprocess_autoencoder(frame):
    img = cv2.resize(frame, (128,128))
    img = img.astype("float32")/255
    return np.expand_dims(img,0)

def preprocess_cnn(frame):
    img = cv2.resize(frame, (224,224))
    img = img.astype("float32")/255
    return np.expand_dims(img,0)

# ---------------- DETECTION ----------------
def detect(upload_img, webcam_img, send_alert, spray):
    if upload_img is not None:
        image = upload_img
    elif webcam_img is not None:
        image = webcam_img
    else:
        return None,"No Image",0,"","",pd.DataFrame(history)

    frame = np.array(image)
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
    annotated = frame.copy()

    # Autoencoder anomaly detection
    ae_input = preprocess_autoencoder(frame)
    reconstructed = autoencoder.predict(ae_input, verbose=0)
    error = np.mean((ae_input - reconstructed)**2)
    THRESHOLD = 0.01

    if error < THRESHOLD:
        label = "healthy"
        confidence = 1.0
    else:
        cnn_input = preprocess_cnn(frame)
        preds = cnn.predict(cnn_input, verbose=0)[0]
        idx = np.argmax(preds)
        label = class_names[idx]
        confidence = float(preds[idx])

    # Annotate image
    cv2.putText(
        annotated,
        f"{label} {confidence:.2f}",
        (20,40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0,255,0),
        3
    )
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    eng = fertilizer[label]["english"]
    tam = fertilizer[label]["tamil"]

    # ---------------- ARDUINO SPRAY ----------------
    if spray:
        if label != "healthy":
            trigger_spray("TRIGGER_HIGH")  # Disease detected
        else:
            trigger_spray("TRIGGER_LOW")   # Healthy

    # ---------------- TELEGRAM ALERT ----------------
    if send_alert and label != "healthy":
        msg = f"""
🌿 SMART COTTON AI ALERT 🌿

Disease: {label}
Confidence: {confidence:.2f}

Recommendation:
English: {eng}
Tamil: {tam}
"""
        send_telegram(msg, annotated)

    # ---------------- HISTORY ----------------
    history.append({
        "Time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Disease": label,
        "Confidence": round(confidence,2)
    })

    return annotated, label, confidence, eng, tam, pd.DataFrame(history)

# ---------------- SETTINGS ----------------
def save_settings(chat, com):
    settings["telegram_chat_id"] = chat
    settings["arduino_com_port"] = com
    connect_arduino()
    return "Settings Saved ✔"

# ---------------- COLORFUL UI STYLE ----------------
css="""
.gradio-container{
background:linear-gradient(120deg,#ff9a9e,#fad0c4,#a1c4fd,#c2e9fb);
font-family:Segoe UI;
}
#title{
text-align:center;
font-size:60px;
font-weight:bold;
color:white;
margin-bottom:20px;
}
.gr-button{
background:linear-gradient(90deg,#ff758c,#ff7eb3);
color:white;
font-weight:bold;
border-radius:12px;
font-size:18px;
}
"""

# ---------------- UI ----------------
with gr.Blocks(css=css) as demo:
    gr.HTML("<div id='title'>🌿 SMART COTTON AI 🌿</div>")

    with gr.Tabs():

        # Detection Tab
        with gr.Tab("Detection"):
            with gr.Row():
                img_input = gr.Image(type="pil", label="Upload Cotton Image")
                webcam = gr.Image(source="webcam", type="pil", label="Webcam Capture")

            send_alert = gr.Checkbox(label="Send Telegram Alert", value=True)
            spray = gr.Checkbox(label="Activate Arduino Spray")

            detect_btn = gr.Button("Run Detection")
            img_output = gr.Image(label="Predicted Image")
            disease = gr.Textbox(label="Disease")
            conf = gr.Textbox(label="Confidence")
            eng = gr.Textbox(label="Recommendation English")
            tam = gr.Textbox(label="Recommendation Tamil")
            history_table = gr.Dataframe(label="Detection History")

            detect_btn.click(
                detect,
                inputs=[img_input, webcam, send_alert, spray],
                outputs=[img_output, disease, conf, eng, tam, history_table]
            )

        # Settings Tab
        with gr.Tab("Settings"):
            chat = gr.Textbox(label="Telegram Chat ID")
            com = gr.Textbox(label="Arduino COM Port", value="COM3")
            save_btn = gr.Button("Save Settings")
            status = gr.Textbox()
            save_btn.click(save_settings, inputs=[chat, com], outputs=status)

        # Telegram Info Tab
        with gr.Tab("Telegram Info"):
            gr.Markdown("""
# 📱 Smart Cotton AI Telegram Bot Guide

## 🌐 English Instructions

### Step 1: Open Bot
👉 Click here to open the bot: https://t.me/CottonAIShield_bot  
OR search **Smart Cotton AI Bot** in Telegram

### Step 2: Start the Bot
- Tap **Start**

### Step 3: Get Your Chat ID
To receive alerts, you need your Chat ID.

👉 Open this helper bot: https://t.me/userinfo3bot  
- Start the bot  
- It will show your **User ID / Chat ID**  

📌 Copy that ID

### Step 4: Add in Settings
- Paste your Chat ID in the **Settings tab**
- Save

✅ Now you will receive **AI-based crop alerts automatically**

---
## 🌾 தமிழ் வழிமுறை (Tamil Instructions)

### படி 1: Bot திறக்கவும்
👉 இந்த link-ஐ கிளிக் செய்யவும்: https://t.me/CottonAIShield_bot  
அல்லது Telegram-ல் **Smart Cotton AI Bot** என்று தேடவும்

### படி 2: Bot தொடங்கவும்
- **Start** பொத்தானை அழுத்தவும்  

### படி 3: Chat ID பெறுவது எப்படி
Alert பெற Chat ID தேவைப்படும்  

👉 இந்த bot-ஐ திறக்கவும்: https://t.me/userinfo3bot  
- Start செய்யவும்  
- உங்கள் **User ID / Chat ID** காட்டப்படும்  

📌 அந்த எண்ணை copy செய்யவும்  

### படி 4: Settings-ல் சேமிக்கவும்
- உங்கள் Chat ID-ஐ Settings tab-ல் paste செய்யவும்  
- Save செய்யவும்  

✅ இப்போது நீங்கள் **AI crop alerts automatically** பெறுவீர்கள்  

---

💡 Note:
- Chat ID என்பது ஒரு unique number  
- இது ஒருமுறை எடுத்தால் போதும்  
""")
demo.launch(share=True)
from telegram import Update
from telegram.ext import ApplicationBuilder, MessageHandler, filters, ContextTypes

async def handle_image(update: Update, context: ContextTypes.DEFAULT_TYPE):
    photo = update.message.photo[-1]
    file = await photo.get_file()
    path = "temp.jpg"
    await file.download_to_drive(path)

    img = cv2.imread(path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    result = detect(img_rgb, None, False, False)

    label = result[1]
    confidence = result[2]

    await update.message.reply_text(
        f"🌿 Result:\n{label}\nConfidence: {confidence:.2f}"
    )

def run_bot():
    app = ApplicationBuilder().token(settings["telegram_bot_id"]).build()
    app.add_handler(MessageHandler(filters.PHOTO, handle_image))
    print("Bot running...")
    app.run_polling()

Loading AI models...
Models Loaded ✔
Running on local URL:  http://127.0.0.1:7862
IMPORTANT: You are using gradio version 3.45.0, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://0b51813f542077b1c2.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
